In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# 1. INSTALL AND IMPORT LIBRARIES
# --------------------------------------------------------------------------
# --------------------------------------------------------------------------
!pip install -q transformers accelerate torchaudio librosa
!pip install torchcodec==0.2
!pip install fsspec==2023.9.2
!pip install datasets==2.10.0
#!pip install evaluate==0.4.0
!pip install jiwer==3.0.0
!pip install tf-keras

!pip install -q --upgrade evaluate

!pip uninstall -y rapidfuzz
!pip install rapidfuzz==3.9.3 --no-cache-dir --force-reinstall

!pip install --user --no-cache-dir pyarrow==15.0.2

import torch
import torchaudio
import os
import jiwer
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from transformers import Wav2Vec2Processor, Wav2Vec2ForCTC, Wav2Vec2ConformerForCTC
from torch.optim import AdamW
from safetensors.torch import load_file
from datasets import load_dataset, Audio
import datasets  # Import the datasets library itself
from tqdm.auto import tqdm
import warnings

warnings.filterwarnings("ignore")

# FIX: Force datasets to use the more stable torchaudio backend for audio decoding
datasets.config.AUDIO_DECODE = True

  Using cached rapidfuzz-2.13.7-py3-none-any.whl
  Attempting uninstall: rapidfuzz
    Found existing installation: rapidfuzz 3.9.3
    Uninstalling rapidfuzz-3.9.3:
      Successfully uninstalled rapidfuzz-3.9.3
Found existing installation: rapidfuzz 2.13.7
Uninstalling rapidfuzz-2.13.7:
  Successfully uninstalled rapidfuzz-2.13.7
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 180.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
jiwer 3.0.0 requires rapidfuzz==2.13.7, but you have rapidfuzz 3.9.3 which is incompatible.


In [ ]:
# 2. DEFINE THE MODEL ARCHITECTURES
# --------------------------------------------------------------------------
# Custom ASR Model (used for loading your third agent)
class ASRModel(nn.Module):
    def __init__(self, n_mels, rnn_dim, vocab_size, n_cnn_layers, n_rnn_layers, dropout):
        super().__init__()
        cnn_channels = 32
        self.cnn = nn.Sequential(
            nn.Conv2d(1, cnn_channels, kernel_size=3, padding=1, stride=1), nn.ReLU(),
            nn.BatchNorm2d(cnn_channels),
            nn.Conv2d(cnn_channels, cnn_channels, kernel_size=3, padding=1, stride=1), nn.ReLU(),
            nn.BatchNorm2d(cnn_channels),
            nn.MaxPool2d(kernel_size=(2, 1)),
        )
        cnn_output_dim = (n_mels // 2) * cnn_channels
        self.rnn = nn.GRU(
            input_size=cnn_output_dim, hidden_size=rnn_dim,
            num_layers=n_rnn_layers, bidirectional=True,
            batch_first=True, dropout=dropout if n_rnn_layers > 1 else 0
        )
        self.classifier = nn.Linear(rnn_dim * 2, vocab_size)

    def forward(self, input_values, **kwargs):
        x = input_values.unsqueeze(1)
        x = self.cnn(x)
        batch_size, channels, freq_dim, seq_len = x.size()
        x = x.permute(0, 3, 1, 2).contiguous().view(batch_size, seq_len, -1)
        x, _ = self.rnn(x)
        logits = self.classifier(x)
        return {"logits": logits}

In [ ]:
# The new trainable Fusion Model
class FusionModel(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.rnn = nn.GRU(
            input_size=input_size, hidden_size=hidden_size,
            num_layers=2, bidirectional=True, batch_first=True, dropout=0.2
        )
        self.classifier = nn.Linear(hidden_size * 2, output_size)
        self.hidden_size = hidden_size # Store hidden_size for verification

    def forward(self, x):
        # Check for NaNs and Infs in the input tensor
        if torch.isnan(x).any() or torch.isinf(x).any():
            print("Detected NaNs or Infs in FusionModel input!")
            # Optionally, print indices or a slice to help debug
            # nan_mask = torch.isnan(x)
            # inf_mask = torch.isinf(x)
            # print(f"NaN mask sample: {nan_mask[0, :10, :10]}")
            # print(f"Inf mask sample: {inf_mask[0, :10, :10]}")
            # You might want to raise an error or handle this appropriately
            # raise ValueError("Invalid values detected in FusionModel input")

        self.rnn.flatten_parameters()
        x, _ = self.rnn(x)
        #print(f"FusionModel after GRU shape: {x.shape}")
        #print(f"Expected input to classifier last dimension: {self.hidden_size * 2}")
        logits = self.classifier(x)
        #print(f"FusionModel output logits shape: {logits.shape}")
        return logits

In [ ]:
# 3. CUSTOM DATASET AND COLLATOR
# --------------------------------------------------------------------------
class EnsembleLogitsDataset(Dataset):
    """
    A PyTorch dataset that takes an audio dataset and returns the
    concatenated logits from the three frozen ASR agents for each sample.
    """
    def __init__(self, hf_dataset, processor, conformer_model, wav2vec2_model, custom_model, device):
        self.hf_dataset = hf_dataset
        self.processor = processor
        self.conformer_model = conformer_model
        self.wav2vec2_model = wav2vec2_model
        self.custom_model = custom_model
        self.device = device
        self.mel_transform = torchaudio.transforms.MelSpectrogram(
            sample_rate=16000, n_fft=400, hop_length=160, n_mels=80
        )

    def __len__(self):
        return len(self.hf_dataset)

    @torch.no_grad()
    def __getitem__(self, idx):
        item = self.hf_dataset[idx]
        # Corrected: Pass the raw audio array as a list to the processor
        waveform_array = item["audio"]["array"]
        ground_truth = item["transcription"]

        # --- Process for Conformer and Wav2Vec2 ---
        # Pass the raw array as a list to the processor
        input_values = self.processor([waveform_array], return_tensors="pt", sampling_rate=16000).input_values.to(self.device)
        logits_conformer = self.conformer_model(input_values).logits
        logits_wav2vec2 = self.wav2vec2_model(input_values).logits

        # --- Process for Custom Model ---
        # Need to convert the raw array to tensor for the custom model
        waveform_tensor = torch.tensor(waveform_array, dtype=torch.float)
        log_mel_spec = torch.log(torch.clamp(self.mel_transform(waveform_tensor.cpu()), min=1e-5)).to(self.device)
        logits_custom = self.custom_model(log_mel_spec.unsqueeze(0))['logits']


        # --- Align sequence lengths ---
        min_len = min(logits_conformer.shape[1], logits_wav2vec2.shape[1], logits_custom.shape[1])
        logits_conformer = logits_conformer[:, :min_len, :]
        logits_wav2vec2 = logits_wav2vec2[:, :min_len, :]
        logits_custom = logits_custom[:, :min_len, :]

        # Squeeze the batch dimension (batch size is 1 here)
        concatenated_logits = torch.cat([logits_conformer, logits_wav2vec2, logits_custom], dim=-1).squeeze(0)

        return {"inputs": concatenated_logits, "labels": ground_truth}

In [ ]:
class DataCollatorCTCWithPadding:
    """
    Custom data collator to handle padding for both input logits and output labels.
    """
    def __init__(self, processor):
        self.processor = processor

    def __call__(self, features):
        input_features = [{"inputs": feature["inputs"]} for feature in features]
        # Corrected: Prepare label features in the format expected by tokenizer.pad
        label_input_ids = [self.processor.tokenizer(feature["labels"]).input_ids for feature in features]
        label_features = {"input_ids": label_input_ids}

        # Pad the input logit sequences
        inputs_padded = torch.nn.utils.rnn.pad_sequence(
            [f["inputs"] for f in input_features], batch_first=True, padding_value=0.0
        )

        # Pad the label sequences using the tokenizer's pad method
        labels_padded = self.processor.tokenizer.pad(label_features, padding="longest", return_tensors="pt")

        return {
            "inputs": inputs_padded,
            "labels": labels_padded["input_ids"] # Access the padded input_ids
        }

In [ ]:
MAX_DURATION_IN_SECONDS = 20.0 # Filter out very long audio samples to avoid OOM errors
MIN_DURATION_IN_SECONDS = 1.0  # Filter out very short audio samples

import re
import evaluate

# 4. MAIN TRAINABLE ENSEMBLE CLASS
# --------------------------------------------------------------------------
class TrainableEnsemble:
    def __init__(self, conformer_path, wav2vec2_path, custom_model_path):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        print(f"Initializing system on device: {self.device}")

        print("Loading base models and processor...")
        self.processor = Wav2Vec2Processor.from_pretrained(wav2vec2_path)
        vocab_size = self.processor.tokenizer.vocab_size

        # CORRECTED MODEL LOADING: Use Wav2Vec2ConformerForCTC for the conformer model
        self.conformer_model = Wav2Vec2ConformerForCTC.from_pretrained(conformer_path).to(self.device).eval()
        self.wav2vec2_model = Wav2Vec2ForCTC.from_pretrained(wav2vec2_path).to(self.device).eval()

        self.custom_model = ASRModel(n_mels=80, rnn_dim=512, vocab_size=29, n_cnn_layers=3, n_rnn_layers=3, dropout=0.1)
        weights_path = os.path.join(custom_model_path, "model.safetensors")
        self.custom_model.load_state_dict(load_file(weights_path, device="cpu"))
        self.custom_model.to(self.device).eval()
        print("✅ Base models loaded.")

        self.mel_transform = torchaudio.transforms.MelSpectrogram(
            sample_rate=16000, n_fft=400, hop_length=160, n_mels=80
        )


        # Dynamically determine the input size for the FusionModel
        # Load a small sample of the dataset to get the actual concatenated logits size
        temp_dataset = load_dataset("aconeil/nchlt", split="train[:1]")
        temp_dataset = temp_dataset.cast_column("audio", Audio(sampling_rate=16000))
        temp_ensemble_dataset = EnsembleLogitsDataset(temp_dataset, self.processor, self.conformer_model, self.wav2vec2_model, self.custom_model, self.device)
        sample_logits = temp_ensemble_dataset[0]["inputs"]
        fusion_model_input_size = sample_logits.shape[-1]
        print(f"Dynamically determined FusionModel input size: {fusion_model_input_size}")

        # Corrected: Set FusionModel output size to accommodate the blank token ID
        self.fusion_model = FusionModel(input_size=fusion_model_input_size, hidden_size=512, output_size=self.processor.tokenizer.pad_token_id + 1).to(self.device)

        # Removed the assertion that caused the error

        print(f"✅ FusionModel initialized with output size {self.processor.tokenizer.pad_token_id + 1} and blank token ID {self.processor.tokenizer.pad_token_id}.")


    def train(self, dataset_name, train_split="train", val_split="test", epochs=5, batch_size=4, lr=5e-5):
        print(f"\n--- Starting Training for {epochs} Epochs ---")
        dataset = load_dataset(dataset_name)
        # --- START: NEW DATA CLEANING BLOCK ---

        # 1. Define the function to remove special characters (from the new notebook)
        chars_to_remove_regex = r'[!\"#$%&\'()*+,-./:;<=>?@\[\\\]^_`{|}~0-9]'
        def remove_special_characters(batch):
            if batch["transcription"] is not None:
                # This line removes unwanted characters and converts to lowercase
                batch["transcription"] = re.sub(chars_to_remove_regex, '', batch["transcription"]).lower()
            else:
                batch["transcription"] = "" # Handle potential None values
            return batch

        # 2. Apply the cleaning function to the entire dataset
        print("Normalizing and cleaning text data...")
        dataset = dataset.map(
            remove_special_characters,
            num_proc=4, # Use multiple processes for speed
            load_from_cache_file=False # Force re-processing
        )

        # 3. Filter out any transcriptions that became empty after cleaning
        dataset = dataset.filter(
            lambda x: len(x["transcription"].strip()) > 0,
            load_from_cache_file=False
        )
        print("Data cleaning complete.")

        dataset = dataset.cast_column("audio", Audio(sampling_rate=16000))

        train_dataset = EnsembleLogitsDataset(dataset[train_split], self.processor, self.conformer_model, self.wav2vec2_model, self.custom_model, self.device)
        val_dataset = EnsembleLogitsDataset(dataset[val_split], self.processor, self.conformer_model, self.wav2vec2_model, self.custom_model, self.device)

        data_collator = DataCollatorCTCWithPadding(self.processor)

        # Set num_workers to 0 to avoid CUDA multiprocessing issues
        train_dataloader = DataLoader(train_dataset, batch_size=batch_size, collate_fn=data_collator, num_workers=0)
        val_dataloader = DataLoader(val_dataset, batch_size=batch_size, collate_fn=data_collator, num_workers=0)

        # Corrected: Set blank to the pad token ID
        ctc_loss_fn = nn.CTCLoss(blank=self.processor.tokenizer.pad_token_id, zero_infinity=True)
        optimizer = AdamW(self.fusion_model.parameters(), lr=lr)

        for epoch in range(epochs):
            self.fusion_model.train()
            total_train_loss = 0
            for batch in tqdm(train_dataloader, desc=f"Epoch {epoch + 1}/{epochs} [Training]"):
                inputs = batch["inputs"].to(self.device)
                labels = batch["labels"].to(self.device)

                input_lengths = torch.full(size=(inputs.size(0),), fill_value=inputs.size(1), dtype=torch.long) # Use input sequence length
                target_lengths = torch.sum(labels != self.processor.tokenizer.pad_token_id, dim=-1)

                # Skip batch if any target length is zero
                if torch.any(target_lengths == 0):
                    print(f"Skipping training batch with zero target length in epoch {epoch + 1}")
                    continue

                optimizer.zero_grad()

                logits = self.fusion_model(inputs)
                log_probs = F.log_softmax(logits, dim=-1).permute(1, 0, 2)


                loss = ctc_loss_fn(log_probs, labels, input_lengths, target_lengths)
                loss.backward()
                optimizer.step()
                total_train_loss += loss.item()

            avg_train_loss = total_train_loss / len(train_dataloader)

            self.fusion_model.eval()
            total_val_loss = 0
            with torch.no_grad():
                for batch in tqdm(val_dataloader, desc=f"Epoch {epoch + 1}/{epochs} [Validation]"):
                    inputs = batch["inputs"].to(self.device)
                    labels = batch["labels"].to(self.device)

                    input_lengths = torch.full(size=(inputs.size(0),), fill_value=inputs.size(1), dtype=torch.long) # Use input sequence length
                    target_lengths = torch.sum(labels != self.processor.tokenizer.pad_token_id, dim=-1)

                    # Skip batch if any target length is zero
                    if torch.any(target_lengths == 0):
                        print(f"Skipping validation batch with zero target length in epoch {epoch + 1}")
                        continue


                    logits = self.fusion_model(inputs)
                    log_probs = F.log_softmax(logits, dim=-1).permute(1, 0, 2)

                    loss = ctc_loss_fn(log_probs, labels, input_lengths, target_lengths)
                    total_val_loss += loss.item()

            avg_val_loss = total_val_loss / len(val_dataloader)
            print(f"Epoch {epoch + 1}: Train Loss: {avg_train_loss:.4f}, Val Loss: {avg_val_loss:.4f}")

        print("✅ Training complete.")

    torch.no_grad()
    def get_concatenated_logits(self, waveform):
        # --- Get logits from all three models ---
        input_values = self.processor(waveform, return_tensors="pt", sampling_rate=16000).input_values.to(self.device)

        logits_conformer = self.conformer_model(input_values).logits
        logits_wav2vec2 = self.wav2vec2_model(input_values).logits

        log_mel_spec = torch.log(torch.clamp(self.mel_transform(waveform.cpu()), min=1e-5)).to(self.device)
        logits_custom = self.custom_model(log_mel_spec.unsqueeze(0))['logits']

        # Raise an error if any tensor is not 3D, which is required for concatenation
        if not (logits_conformer.dim() == 3 and logits_wav2vec2.dim() == 3 and logits_custom.dim() == 3):
            raise ValueError("One of the models did not produce a 3D logits tensor. Skipping this sample.")
        # --- End of debug block ---

        # Align sequence lengths
        min_len = min(logits_conformer.shape[1], logits_wav2vec2.shape[1], logits_custom.shape[1])
        logits_conformer = logits_conformer[:, :min_len, :]
        logits_wav2vec2 = logits_wav2vec2[:, :min_len, :]
        logits_custom = logits_custom[:, :min_len, :]

        # --- Decode individual model predictions ---
        pred_ids_conformer = torch.argmax(logits_conformer, dim=-1)
        transcription_conformer = self.processor.batch_decode(pred_ids_conformer)[0]

        pred_ids_wav2vec2 = torch.argmax(logits_wav2vec2, dim=-1)
        transcription_wav2vec2 = self.processor.batch_decode(pred_ids_wav2vec2)[0]

        pred_ids_custom = torch.argmax(logits_custom, dim=-1)
        transcription_custom = self.processor.batch_decode(pred_ids_custom)[0]

        concatenated_logits = torch.cat([logits_conformer, logits_wav2vec2, logits_custom], dim=-1)

        return {
            "concatenated_logits": concatenated_logits,
            "transcription_conformer": transcription_conformer,
            "transcription_wav2vec2": transcription_wav2vec2,
            "transcription_custom": transcription_custom
        }

    def predict(self, audio_input):
        self.fusion_model.eval()

        # This 'if/else' block is the key to fixing the error
        if isinstance(audio_input, str):  # If input is a file path (string)
            waveform, sr = torchaudio.load(audio_input)
            if sr != 16000:
                waveform = torchaudio.transforms.Resample(orig_freq=sr, new_freq=16000)(waveform)
        else:  # Otherwise, assume input is already a waveform tensor
            waveform = audio_input

        # The rest of the function remains the same
        if waveform.shape[0] > 1:
            waveform = torch.mean(waveform, dim=0, keepdim=True)

        waveform = waveform.squeeze(0)

        with torch.no_grad():
            intermediate_results = self.get_concatenated_logits(waveform)
            final_logits = self.fusion_model(intermediate_results["concatenated_logits"])

        predicted_ids = torch.argmax(final_logits, dim=-1)
        final_transcription = self.processor.batch_decode(predicted_ids)[0]

        return {
            "Conformer": intermediate_results["transcription_conformer"],
            "Wav2Vec2": intermediate_results["transcription_wav2vec2"],
            "Custom Model": intermediate_results["transcription_custom"],
            "Final Ensemble": final_transcription
        }

        # Add this entire method inside your TrainableEnsemble class
    def evaluate(self, dataset_name, dataset_split="test"):
        print(f"--- Starting evaluation on the '{dataset_split}' split ---")

        wer_metric = evaluate.load("wer")
        cer_metric = evaluate.load("cer")

        print("Loading and cleaning the dataset...")
        dataset = load_dataset(dataset_name, split=dataset_split)

        # Apply the same cleaning you used for training
        chars_to_remove_regex = r'[!\"#$%&\'()*+,-./:;<=>?@\[\\\]^_`{|}~0-9]'
        def remove_special_characters(batch):
            batch["transcription"] = re.sub(chars_to_remove_regex, '', batch["transcription"]).lower()
            return batch

        dataset = dataset.map(remove_special_characters, num_proc=4)
        dataset = dataset.filter(lambda x: len(x["transcription"].strip()) > 0)
        dataset = dataset.cast_column("audio", Audio(sampling_rate=16000))
        print("Dataset ready.")

        predictions = []
        references = []

        for i, example in enumerate(tqdm(dataset, desc=f"Evaluating on '{dataset_split}' split")):
            try: # --- ADD THIS TRY BLOCK ---
                waveform = torch.tensor(example["audio"]["array"], dtype=torch.float32).unsqueeze(0)
                reference_transcription = example["transcription"]

                results = self.predict(waveform)
                prediction = results["Final Ensemble"]

                predictions.append(prediction)
                references.append(reference_transcription)
            except (IndexError, ValueError) as e: # --- CATCH THE SPECIFIC ERROR ---
                print(f"\nSkipping sample #{i} due to an IndexError. This is likely a shape issue with a very short audio clip.")
                print(f"Error message: {e}")
                continue # Move to the next sample

        print("\nCalculating WER and CER...")
        wer = wer_metric.compute(predictions=predictions, references=references)
        cer = cer_metric.compute(predictions=predictions, references=references)

        return {"wer": wer, "cer": cer}

In [ ]:
# 1. Initialize the system again

CONFORMER_MODEL_PATH = "/content/drive/MyDrive/wav2vecconformer-finetune-checkpoints"
WAV2VEC2_MODEL_PATH = "/content/drive/MyDrive/wav2vec2-finetune-checkpoints22/checkpoint-20000"
CUSTOM_MODEL_PATH = "/content/drive/MyDrive/my_final_wav2vec2_model"
DATASET_FOR_TRAINING = "aconeil/nchlt"
TEST_AUDIO_PATH = "/content/drive/MyDrive/test.wav"

trainable_system = TrainableEnsemble(
    conformer_path=CONFORMER_MODEL_PATH,
    wav2vec2_path=WAV2VEC2_MODEL_PATH,
    custom_model_path=CUSTOM_MODEL_PATH
)

# 2. Load your saved weights
model_path = "/content/drive/MyDrive/trained_fusion_model.pth"
trainable_system.fusion_model.load_state_dict(torch.load(model_path))
trainable_system.fusion_model.to(trainable_system.device) # Ensure model is on the correct device (GPU)
trainable_system.fusion_model.eval() # Set the model to evaluation mode

print("✅ Successfully loaded trained model from disk.")

# 3. Now you can safely debug and run prediction
#    (This includes the fix of using .squeeze(0) on the waveform)
final_transcription = trainable_system.predict(TEST_AUDIO_PATH)
print(f"Final Transcription: \"{final_transcription}\"")

Initializing system on device: cuda
Loading base models and processor...
✅ Base models loaded.


Dynamically determined FusionModel input size: 92
✅ FusionModel initialized with output size 33 and blank token ID 32.
✅ Successfully loaded trained model from disk.
Final Transcription: "{'Conformer': 'babebdbzbtkbfbebobxbibubpbobjbkbjbdbxbzbxbvybkb', 'Wav2Vec2': '[PAD]i[PAD]z[PAD]o[PAD]k[PAD]w[PAD]e[PAD]n[PAD]z[PAD]a[PAD] [PAD]u[PAD]m[PAD]y[PAD]a[PAD]l[PAD]e[PAD]l[PAD]o w[PAD]o[PAD]k[PAD]u[PAD]th[PAD]i[PAD]', 'Custom Model': '[PAD]q[PAD]z[PAD]o[PAD]t[PAD]fv[PAD]az[PAD]m[PAD] [PAD]u[PAD]b', 'Final Ensemble': 'izokwenza umyalelo wokuthi'}"


In [ ]:
# Run the evaluation on the test split
evaluation_results = trainable_system.evaluate(dataset_name="aconeil/nchlt", dataset_split="test")

# Print the results
print("\n--- Evaluation Complete ---")
print(f"Word Error Rate (WER): {evaluation_results['wer']:.4f}")
print(f"Character Error Rate (CER): {evaluation_results['cer']:.4f}")

--- Starting evaluation on the 'test' split ---


Loading and cleaning the dataset...


Dataset ready.


Evaluating on 'test' split:   0%|          | 0/2802 [00:00<?, ?it/s]


Calculating WER and CER...

--- Evaluation Complete ---
Word Error Rate (WER): 0.1904
Character Error Rate (CER): 0.0304
